# ⚡ Google Drive to Google Drive Cloud GUI Transfer
Transfer files and entire folders directly from one Google Drive account to another at ultra-high speed (100–250 MB/s) using Google Cloud servers, bypassing your home internet limits.

---

### 🚀 How It Works
1. **Step 1**: Connect your **Destination Google Drive** (Account 2).
2. **Step 2**: Install high-speed transfer engine.
3. **Step 2.5 (Optional)**: Activate Anti-Disconnect Keep-Alive.
4. **Step 3**: Paste your Google Drive file or folder links (from Account 1), pick your destination folder, and click **⚡ Start Transfer**!

> 💡 **Tip:** Make sure the link from Account 1 has sharing set to **"Anyone with the link can view"** (or shared directly with Account 2) so Google Colab can duplicate it.

### Step 1: Connect your Google Drive
Run this cell below, click the authorization link, and sign in to your **Destination Google Drive Account**.

In [ ]:
from google.colab import drive
import os

gdrive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"
if not os.path.exists(gdrive_root):
    print("[INFO] Connecting to Google Drive...")
    drive.mount('/content/drive')
    print("\n[SUCCESS] Google Drive successfully mounted!")
else:
    print("[INFO] Google Drive is already mounted and ready to use!")


### Step 2: Install High-Speed Transfer Engine
Run this cell to update the `gdown` high-speed Google Drive transfer engine. (Only needs to run once per session).

In [ ]:
!pip install --upgrade -q gdown
print("[SUCCESS] Transfer engine installed and up-to-date!")


### Step 2.5: Anti-Disconnect Keep-Alive (Optional)
Run this cell if you are transferring large files/folders and want to prevent Google Colab from disconnecting or going to sleep.

In [ ]:
#@title Run Anti-Disconnect Keep-Alive
from IPython.display import display, HTML

display(HTML('''
<audio src="data:audio/wav;base64,UklGRiQAAABXQVZFZm10IBAAAAABAAEARKwAAIhYAQACABAAZGF0YQAAAAA=" autoplay loop></audio>
<script>
function KeepClicking(){
   console.log("⚡ Colab Keep-Alive Active: " + new Date().toLocaleTimeString());
   let btn = document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect") || 
             document.querySelector("#top-toolbar > colab-connect-button") ||
             document.querySelector("colab-toolbar-button#connect");
   if (btn) { btn.click(); }
}
setInterval(KeepClicking, 60000);
</script>
<div style="padding: 12px; background-color: #e6f4ea; border-left: 4px solid #34a853; border-radius: 6px; color: #137333; font-weight: 500; font-family: sans-serif;">
   🟢 <strong>Anti-Disconnect Active:</strong> Silent background audio and automated pinger are preventing this tab from sleeping.
</div>
'''))


### Step 3: Interactive Transfer Queue (With Movie/Subfolder Support)
Use the interactive panel below to queue Google Drive files or entire folders:
1. **Download Link(s)**: Paste any Google Drive link (`drive.google.com/file/d/...` or `drive.google.com/drive/folders/...`).
2. **Auto-Pull Name**: Click **🪄 Auto-Pull Name** to automatically extract the folder/movie name from the link.
3. **Destination**: Select an existing movie folder from the dropdown or type a new one, plus optional inside subfolders.
4. **Transfer**: Click **⚡ Start Transfer** to copy at internal Google Cloud speed (100–250 MB/s) with live progress!

In [ ]:
#@title Interactive Transfer Queue Manager (Google Drive to Google Drive)
# Run this cell to open the Interactive Link Queue Manager

import os
import subprocess
import time
import sys
import re
import fcntl
import pty
import requests
import urllib.parse
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

def get_gdrive_root():
    for candidate in ["/content/drive/MyDrive", "/content/drive/My Drive"]:
        if os.path.exists(candidate):
            return candidate
    return None

if 'gdrive_link_queue' not in globals():
    gdrive_link_queue = []

def classify_gdrive_link(link):
    if not link:
        return None, None
    clean = link.strip().rstrip('.,;')
    m_folder = re.search(r'drive\.google\.com/drive/(?:u/\d+/)?folders/([a-zA-Z0-9_-]+)', clean)
    if m_folder:
        return m_folder.group(1), 'folder'
    m_file = re.search(r'drive\.google\.com/file/d/([a-zA-Z0-9_-]+)', clean)
    if m_file:
        return m_file.group(1), 'file'
    m_open = re.search(r'drive\.google\.com/open\?id=([a-zA-Z0-9_-]+)', clean)
    if m_open:
        return m_open.group(1), 'file'
    m_uc = re.search(r'drive\.google\.com/uc\?(?:[^&]*&)*id=([a-zA-Z0-9_-]+)', clean)
    if m_uc:
        return m_uc.group(1), 'file'
    if re.match(r'^[a-zA-Z0-9_-]{25,50}$', clean):
        return clean, 'unknown'
    return None, None

def extract_gdrive_links(text):
    if not text:
        return []
    clean_text = text.replace(',', ' ').replace('"', ' ').replace("'", ' ')
    candidates = clean_text.split()
    clean_links = []
    for token in candidates:
        token = token.strip().rstrip('.,;')
        fid, ftype = classify_gdrive_link(token)
        if fid:
            if not token.startswith('http://') and not token.startswith('https://'):
                token = 'https://drive.google.com/open?id=' + fid
            if token not in clean_links:
                clean_links.append(token)
    return clean_links

def get_gdrive_title(link):
    if not link:
        return None
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    if not link.startswith('http://') and not link.startswith('https://'):
        link = 'https://' + link
    try:
        res = requests.get(link, headers=headers, stream=True, timeout=6)
        content = b''
        for chunk in res.iter_content(chunk_size=4096):
            content += chunk
            if len(content) > 60000:
                break
        res.close()
        html = content.decode('utf-8', errors='ignore')
        m = re.search(r'<title>(.*?)</title>', html, re.IGNORECASE)
        if m:
            raw = m.group(1).replace(' - Google Drive', '').strip()
            if raw and 'error' not in raw.lower() and 'not found' not in raw.lower() and 'access denied' not in raw.lower():
                clean = re.sub(r'\.[a-zA-Z0-9]{2,5}$', '', raw)
                return clean.strip()
    except:
        pass
    return None

def sanitize_folder_name(name):
    if not name:
        return ""
    cleaned = name.strip()
    for bad_char in ['<', '>', ':', '"', '/', '\\', '|', '?', '*']:
        cleaned = cleaned.replace(bad_char, '_')
    cleaned = ' '.join(cleaned.split())
    return cleaned.strip(' .')

def resolve_subfolder_path(movie_val, inner_val):
    parts = []
    for raw in [movie_val, inner_val]:
        if raw:
            for segment in raw.replace('\\', '/').split('/'):
                seg_clean = sanitize_folder_name(segment)
                if seg_clean and not (seg_clean.startswith('[') and seg_clean.endswith(']')):
                    parts.append(seg_clean)
    if parts:
        return os.path.join(*parts), " / ".join(parts)
    return "", ""

# UI Controls
# 1. Link & Auto-Name Controls
link_input = widgets.Text(
    value="",
    description="Google Drive Link(s):",
    placeholder="Paste Google Drive File or Folder URL...",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='500px')
)

auto_name_btn = widgets.Button(
    description=" 🪄 Auto-Pull Name",
    icon="magic",
    button_style="info",
    tooltip="Auto-pull folder/movie name from the Google Drive link",
    layout=widgets.Layout(width='150px', height='36px')
)

link_row = widgets.HBox([link_input, auto_name_btn], layout=widgets.Layout(margin='2px 0'))

# 2. Destination Controls
base_folder_input = widgets.Text(
    value="GDrive_Transfers",
    description="Base GDrive Folder:",
    placeholder="Main folder in MyDrive",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='655px')
)

def get_existing_gdrive_folders():
    gdrive_root = get_gdrive_root()
    if not gdrive_root:
        return []
    base_gdrive = sanitize_folder_name(base_folder_input.value.strip()) or "GDrive_Transfers"
    base_path = os.path.join(gdrive_root, base_gdrive)
    if not os.path.exists(base_path):
        return []
    try:
        folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)) and not f.startswith('.')]
        return sorted(folders, key=lambda s: s.lower())
    except Exception:
        return []

def get_existing_inner_subfolders(movie_name):
    gdrive_root = get_gdrive_root()
    if not gdrive_root or not movie_name:
        return []
    base_gdrive = sanitize_folder_name(base_folder_input.value.strip()) or "GDrive_Transfers"
    movie_path = os.path.join(gdrive_root, base_gdrive, movie_name)
    if not os.path.exists(movie_path) or not os.path.isdir(movie_path):
        return []
    try:
        subdirs = [f for f in os.listdir(movie_path) if os.path.isdir(os.path.join(movie_path, f)) and not f.startswith('.')]
        return sorted(subdirs, key=lambda s: s.lower())
    except Exception:
        return []

existing_folders = get_existing_gdrive_folders()

folder_dropdown = widgets.Dropdown(
    options=["[ ➕ New Movie / Type Below ]"] + existing_folders,
    value="[ ➕ New Movie / Type Below ]",
    description="Select Movie Folder:",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='500px')
)

refresh_folders_btn = widgets.Button(
    description=" Refresh",
    icon="refresh",
    button_style="",
    tooltip="Scan Google Drive for existing movie folders",
    layout=widgets.Layout(width='150px', height='36px')
)

folder_select_row = widgets.HBox([folder_dropdown, refresh_folders_btn], layout=widgets.Layout(margin='2px 0'))

subfolder_input = widgets.Text(
    value="",
    description="Movie Folder Name:",
    placeholder="Select from dropdown above or type a new movie name...",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='655px')
)

inner_dropdown = widgets.Dropdown(
    options=["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"],
    value="[ No Inside Folder (Save in Movie Root) ]",
    description="Select Inside Folder:",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='500px')
)

inner_subfolder_input = widgets.Text(
    value="",
    description="Inside Folder Name:",
    placeholder="e.g. Dialogue, Songs, 4K Clips (Optional - leave blank if not needed)",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='655px')
)

def on_folder_dropdown_changed(change):
    selected_movie = change['new']
    if selected_movie and not selected_movie.startswith('['):
        subfolder_input.value = selected_movie
        inners = get_existing_inner_subfolders(selected_movie)
        inner_dropdown.options = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"] + inners
        inner_dropdown.value = "[ No Inside Folder (Save in Movie Root) ]"
        inner_subfolder_input.value = ""
    elif selected_movie == "[ ➕ New Movie / Type Below ]":
        subfolder_input.value = ""
        inner_dropdown.options = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"]
        inner_dropdown.value = "[ No Inside Folder (Save in Movie Root) ]"
        inner_subfolder_input.value = ""

folder_dropdown.observe(on_folder_dropdown_changed, names='value')

def on_subfolder_input_changed(change):
    val = change.get('new', '').strip()
    if 'drive.google.com' in val:
        if not link_input.value.strip():
            link_input.value = val
        title = get_gdrive_title(val)
        if title:
            subfolder_input.value = title
            folder_dropdown.value = "[ ➕ New Movie / Type Below ]"
            with status_output:
                clear_output()
                print(f"\033[92m✨ Auto-detected Movie Folder Name: '{title}'\033[0m")
            return
    if val:
        inners = get_existing_inner_subfolders(val)
        current_opts = list(inner_dropdown.options)
        new_opts = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"] + inners
        if new_opts != current_opts:
            inner_dropdown.options = new_opts

subfolder_input.observe(on_subfolder_input_changed, names='value')

def on_inner_dropdown_changed(change):
    selected_inner = change['new']
    if selected_inner and not selected_inner.startswith('['):
        inner_subfolder_input.value = selected_inner
    elif selected_inner == "[ No Inside Folder (Save in Movie Root) ]":
        inner_subfolder_input.value = ""
    elif selected_inner == "[ ➕ Create New Inside Folder ]":
        inner_subfolder_input.value = ""

inner_dropdown.observe(on_inner_dropdown_changed, names='value')

def on_auto_name_clicked(b=None):
    raw_text = link_input.value.strip()
    sub_text = subfolder_input.value.strip()

    if 'drive.google.com' in sub_text and 'drive.google.com' not in raw_text:
        raw_text = sub_text
        if not link_input.value.strip():
            link_input.value = sub_text
        subfolder_input.value = ""

    if not raw_text:
        with status_output:
            clear_output()
            print("\033[93m[TIP] Paste a Google Drive link into 'Download Link(s)' first, then click 'Auto-Pull Name'.\033[0m")
        return

    first_link = raw_text.splitlines()[0].strip() if '\n' in raw_text else raw_text.split(',')[0].strip()
    with status_output:
        clear_output()
        print("\033[96m[INFO] Extracting folder/file title from Google Drive link...\033[0m")
    title = get_gdrive_title(first_link)
    if title:
        subfolder_input.value = title
        folder_dropdown.value = "[ ➕ New Movie / Type Below ]"
        with status_output:
            clear_output()
            print(f"\033[92m✨ Auto-detected Movie Folder Name: '{title}'\033[0m")
    else:
        with status_output:
            clear_output()
            print("\033[93m[NOTE] Could not auto-detect title from this link. You can type it manually.\033[0m")

auto_name_btn.on_click(on_auto_name_clicked)

def on_link_input_changed(change):
    raw_text = change.get('new', '').strip()
    if raw_text and not subfolder_input.value.strip() and (folder_dropdown.value == "[ ➕ New Movie / Type Below ]" or not folder_dropdown.value):
        first_link = raw_text.splitlines()[0].strip() if '\n' in raw_text else raw_text.split(',')[0].strip()
        title = get_gdrive_title(first_link)
        if title and not subfolder_input.value.strip():
            subfolder_input.value = title
            with status_output:
                clear_output()
                print(f"\033[92m✨ Auto-detected Movie Folder Name from link: '{title}'\033[0m")

link_input.observe(on_link_input_changed, names='value')

def on_refresh_folders_clicked(b=None):
    folders = get_existing_gdrive_folders()
    current = folder_dropdown.value
    folder_dropdown.options = ["[ ➕ New Movie / Type Below ]"] + folders
    if current in folder_dropdown.options:
        folder_dropdown.value = current
    elif subfolder_input.value in folder_dropdown.options:
        folder_dropdown.value = subfolder_input.value
    if subfolder_input.value:
        inners = get_existing_inner_subfolders(subfolder_input.value.strip())
        inner_dropdown.options = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"] + inners
    with status_output:
        clear_output()
        if folders:
            print(f"\033[92m[INFO] Scanned Google Drive: Found {len(folders)} existing folder(s).\033[0m")
        else:
            print(f"\033[93m[INFO] No existing folders found in '{base_folder_input.value.strip()}' yet.\033[0m")

refresh_folders_btn.on_click(on_refresh_folders_clicked)

def on_base_folder_changed(change):
    folders = get_existing_gdrive_folders()
    folder_dropdown.options = ["[ ➕ New Movie / Type Below ]"] + folders
    folder_dropdown.value = "[ ➕ New Movie / Type Below ]"

base_folder_input.observe(on_base_folder_changed, names='value')

add_btn = widgets.Button(
    description=" Add to Queue",
    icon="plus",
    button_style="info",
    layout=widgets.Layout(width='140px', height='36px')
)

remove_btn = widgets.Button(
    description=" Remove Last",
    icon="minus",
    button_style="warning",
    layout=widgets.Layout(width='130px', height='36px')
)

clear_btn = widgets.Button(
    description=" Clear Queue",
    icon="trash",
    button_style="danger",
    layout=widgets.Layout(width='130px', height='36px')
)

reset_btn = widgets.Button(
    description=" Reset Status",
    icon="refresh",
    button_style="",
    layout=widgets.Layout(width='130px', height='36px')
)

start_btn = widgets.Button(
    description=" Start Transfer",
    icon="play",
    button_style="success",
    layout=widgets.Layout(width='170px', height='40px')
)

queue_output = widgets.Output()
status_output = widgets.Output()

def render_queue():
    with queue_output:
        clear_output(wait=True)
        if not gdrive_link_queue:
            display(HTML('''
            <div style="border: 1px dashed #ced4da; border-radius: 8px; padding: 16px; margin: 10px 0; color: #6c757d; text-align: center; background: #f8f9fa;">
                <em>Queue is empty. Specify a folder name (if needed), paste a Google Drive link, and click <strong>Add to Queue</strong>.</em>
            </div>
            '''))
            return
        
        html_rows = ""
        for i, item in enumerate(gdrive_link_queue, 1):
            badge_color = {
                "Pending": "#6c757d",
                "Transferring": "#007bff",
                "Completed": "#28a745",
                "Failed": "#dc3545"
            }.get(item["status"], "#007bff" if "Transferring" in item["status"] else "#6c757d")
            
            subfolder_display = item.get('subfolder_display') or item.get('subfolder', '')
            if subfolder_display:
                folder_tag = f'<span style="background-color: #e8f0fe; color: #1a73e8; border: 1px solid #d2e3fc; padding: 3px 8px; border-radius: 6px; font-weight: 500;">📁 {subfolder_display}</span>'
            else:
                folder_tag = '<span style="color: #adb5bd; font-style: italic;">Root (Base Folder)</span>'
            
            link_type_tag = f'<span style="background-color: #f1f3f4; color: #3c4043; padding: 2px 6px; border-radius: 4px; font-size: 11px; margin-right: 6px; font-weight: 600;">{item.get("type", "file").upper()}</span>'
            
            html_rows += f'''
            <tr style="border-bottom: 1px solid #dee2e6;">
                <td style="padding: 8px 12px; font-weight: bold; width: 35px; text-align: center;">{i}</td>
                <td style="padding: 8px 12px; width: 220px;">{folder_tag}</td>
                <td style="padding: 8px 12px; font-family: monospace; word-break: break-all;">{link_type_tag}{item['link']}</td>
                <td style="padding: 8px 12px; width: 150px; text-align: center;">
                    <span style="background-color: {badge_color}; color: white; padding: 4px 10px; border-radius: 12px; font-size: 12px; font-weight: bold;">
                        {item['status']}
                    </span>
                </td>
            </tr>
            '''
            
        table_html = f'''
        <div style="border: 1px solid #dee2e6; border-radius: 8px; overflow: hidden; margin: 10px 0;">
            <table style="width: 100%; border-collapse: collapse; font-size: 13px; text-align: left;">
                <thead>
                    <tr style="background-color: #f1f3f4; border-bottom: 2px solid #dee2e6;">
                        <th style="padding: 10px 12px; text-align: center;">#</th>
                        <th style="padding: 10px 12px;">Subfolder / Destination</th>
                        <th style="padding: 10px 12px;">Google Drive Link</th>
                        <th style="padding: 10px 12px; text-align: center;">Status</th>
                    </tr>
                </thead>
                <tbody>
                    {html_rows}
                </tbody>
            </table>
        </div>
        '''
        display(HTML(table_html))

def on_add_clicked(b=None):
    raw_text = link_input.value.strip()
    movie_val = subfolder_input.value.strip()
    if not movie_val and folder_dropdown.value and not folder_dropdown.value.startswith('['):
        movie_val = folder_dropdown.value

    # Auto-fallback: if no folder is specified, auto-pull from link
    if not movie_val and raw_text:
        first_link = raw_text.splitlines()[0].strip() if '\n' in raw_text else raw_text.split(',')[0].strip()
        auto_title = get_gdrive_title(first_link)
        if auto_title:
            movie_val = auto_title
            subfolder_input.value = auto_title

    inner_val = inner_subfolder_input.value.strip()
    if not inner_val and inner_dropdown.value and not inner_dropdown.value.startswith('['):
        inner_val = inner_dropdown.value

    subfolder, display_name = resolve_subfolder_path(movie_val, inner_val)
    
    if not raw_text:
        with status_output:
            clear_output()
            print("\033[91m[ERROR] Please paste a valid Google Drive link first!\033[0m")
        return
        
    links = extract_gdrive_links(raw_text)
    if not links:
        candidates = [l.strip() for l in raw_text.split() if l.strip()]
        links = candidates
        
    added_count = 0
    updated_count = 0
    for link in links:
        fid, ftype = classify_gdrive_link(link)
        existing = next((item for item in gdrive_link_queue if item["link"] == link), None)
        if existing:
            existing["subfolder"] = subfolder
            existing["subfolder_display"] = display_name
            existing["status"] = "Pending"
            existing["type"] = ftype or "file"
            updated_count += 1
        else:
            gdrive_link_queue.append({
                "link": link,
                "subfolder": subfolder,
                "subfolder_display": display_name,
                "status": "Pending",
                "type": ftype or "file"
            })
            added_count += 1
            
    link_input.value = ""
    with status_output:
        clear_output()
        parts = []
        if added_count > 0:
            parts.append(f"Added {added_count} new link(s)")
        if updated_count > 0:
            parts.append(f"Reset {updated_count} existing link(s) to 'Pending'")
        folder_info = f" in folder '{display_name}'" if display_name else " in Base folder"
        print(f"\033[92m[INFO] {', '.join(parts)}{folder_info}.\033[0m")
    render_queue()

def on_remove_clicked(b):
    if gdrive_link_queue:
        removed = gdrive_link_queue.pop()
        with status_output:
            clear_output()
            folder_info = f" ({removed.get('subfolder')})" if removed.get('subfolder') else ""
            print(f"\033[93m[INFO] Removed last item: {removed['link']}{folder_info}\033[0m")
        render_queue()
    else:
        with status_output:
            clear_output()
            print("\033[93m[INFO] Queue is already empty.\033[0m")

def on_clear_clicked(b):
    global gdrive_link_queue
    gdrive_link_queue = []
    with status_output:
        clear_output()
        print("\033[93m[INFO] Queue cleared.\033[0m")
    render_queue()

def on_reset_clicked(b):
    for item in gdrive_link_queue:
        item["status"] = "Pending"
    with status_output:
        clear_output()
        print("\033[92m[INFO] All items in queue have been reset to 'Pending'. Ready to transfer!\033[0m")
    render_queue()

def run_transfers(b):
    if not gdrive_link_queue:
        with status_output:
            clear_output()
            print("\033[91m[ERROR] The queue is empty! Add at least one link before starting.\033[0m")
        return
        
    gdrive_root = get_gdrive_root()
    if not gdrive_root:
        with status_output:
            clear_output()
            print("\033[93m[NOTICE] Google Drive not mounted yet. Attempting automatic mount...\033[0m")
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            gdrive_root = get_gdrive_root()
        except:
            pass
            
    if not gdrive_root:
        with status_output:
            clear_output()
            print("\033[91m" + "="*60)
            print("[CRITICAL ERROR] DESTINATION GOOGLE DRIVE IS NOT CONNECTED!")
            print("="*60 + "\033[0m")
            print("\033[93mPlease run 'Step 1: Connect your Google Drive' cell above and authorize your account.\033[0m")
            print("\033[93mWithout Step 1, files cannot be saved to Google Drive!\033[0m")
        return
    
    if all(item["status"] in ("Completed", "Failed") for item in gdrive_link_queue):
        for item in gdrive_link_queue:
            item["status"] = "Pending"
        render_queue()

    add_btn.disabled = True
    remove_btn.disabled = True
    clear_btn.disabled = True
    reset_btn.disabled = True
    start_btn.disabled = True
    link_input.disabled = True
    subfolder_input.disabled = True
    base_folder_input.disabled = True
    folder_dropdown.disabled = True
    refresh_folders_btn.disabled = True
    inner_dropdown.disabled = True
    inner_subfolder_input.disabled = True
    
    base_gdrive = sanitize_folder_name(base_folder_input.value.strip()) or "GDrive_Transfers"
    base_target_dir = os.path.join(gdrive_root, base_gdrive)
    os.makedirs(base_target_dir, exist_ok=True)
    
    with status_output:
        clear_output()
        print(f"\033[94m[INFO] Destination Google Drive: {base_target_dir}\033[0m")
        print(f"\033[94m[INFO] Total links queued: {len(gdrive_link_queue)}\033[0m")
        print("\033[1m[NOTE] Live cloud transfer progress will stream below:\033[0m\n")
        
        successful = 0
        failed = []
        
        for idx, item in enumerate(gdrive_link_queue, 1):
            if item["status"] == "Completed":
                continue
                
            item["status"] = "Transferring"
            render_queue()
            
            page_url = item["link"]
            subfolder = item.get("subfolder", "").strip()
            link_type = item.get("type", "file")
            
            if subfolder:
                target_dir = os.path.join(base_target_dir, subfolder)
                dest_display = f"{base_gdrive} / {subfolder}"
            else:
                target_dir = base_target_dir
                dest_display = base_gdrive
                
            os.makedirs(target_dir, exist_ok=True)
            
            print(f"\n\033[95m{'='*60}\033[0m")
            print(f"\033[94m[TRANSFER {idx}/{len(gdrive_link_queue)}]\033[0m")
            print(f"\033[96m[DESTINATION]\033[0m Google Drive -> {dest_display}")
            print(f"\033[96m[TARGET PATH]\033[0m {target_dir}")
            print(f"\033[96m[SOURCE URL]\033[0m {page_url} ({link_type.upper()})")
            print("\033[92m[GOOGLE CLOUD ENGINE]\033[0m Starting high-speed cloud copy (100–250 MB/s)...")
            print(f"\033[95m{'='*60}\033[0m")
            
            master, slave = pty.openpty()
            
            if link_type == 'folder':
                cmd = ["gdown", "--folder", page_url, "-O", target_dir, "--remaining-ok"]
            else:
                cmd = ["gdown", page_url, "-O", target_dir + "/", "--fuzzy", "--remaining-ok"]
            
            process = subprocess.Popen(cmd, stdout=slave, stderr=slave, text=True, close_fds=True)
            os.close(slave)
            
            fl = fcntl.fcntl(master, fcntl.F_GETFL)
            fcntl.fcntl(master, fcntl.F_SETFL, fl | os.O_NONBLOCK)
            
            buffer = ""
            last_diag_print = 0
            last_pct = -1
            last_table_update_pct = -1
            tqdm_pattern = re.compile(r'(\d+)%\|.*?\|\s*([0-9.]+[A-Za-z]+)/([0-9.]+[A-Za-z]+)\s+\[([0-9:]+)<([0-9:]+),\s*([0-9.]+[A-Za-z/]+)\]')
            
            while True:
                try:
                    chunk = os.read(master, 1024)
                    if chunk:
                        buffer += chunk.decode('utf-8', errors='ignore')
                except OSError:
                    pass
                
                while '\r' in buffer or '\n' in buffer:
                    idx_r = buffer.find('\r')
                    idx_n = buffer.find('\n')
                    cut = min(idx_r, idx_n) if (idx_r != -1 and idx_n != -1) else max(idx_r, idx_n)
                    line = buffer[:cut].strip()
                    buffer = buffer[cut+1:]
                    
                    if not line:
                        continue
                        
                    m = tqdm_pattern.search(line)
                    if m:
                        pct, cur, total, elapsed, eta, speed = m.groups()
                        pct_int = int(pct) if pct and pct.isdigit() else -1
                        now = time.time()
                        
                        # Update status badge in table every ~10%
                        if pct_int != -1 and (pct_int - last_table_update_pct >= 10 or pct_int == 100):
                            last_table_update_pct = pct_int
                            item["status"] = f"Transferring ({pct_int}%)"
                            render_queue()
                            
                        # Print live progress line every 1.5s or on 5% jump
                        if (now - last_diag_print >= 1.5) or (pct_int != -1 and pct_int >= last_pct + 5):
                            last_diag_print = now
                            if pct_int != -1:
                                last_pct = pct_int
                            bar_len = 20
                            filled = int(pct_int / 100 * bar_len)
                            bar = "█" * filled + "░" * (bar_len - filled)
                            prog_msg = f"\033[96m[LIVE SPEED & PROGRESS]\033[0m \033[92m[{bar}] {pct}%\033[0m ({cur} / {total}) | ⚡ Speed: {speed} | ⏳ ETA: {eta}"
                            sys.stdout.write(f"{prog_msg}\n")
                            sys.stdout.flush()
                    else:
                        clean_line = re.sub(r'\x1b\[[0-9;]*[a-zA-Z]', '', line).strip()
                        if clean_line and ('%' not in clean_line):
                            sys.stdout.write(f"{clean_line}\n")
                            sys.stdout.flush()
                
                if process.poll() is not None:
                    try:
                        remaining = os.read(master, 4096)
                        if remaining:
                            buffer += remaining.decode('utf-8', errors='ignore')
                    except:
                        pass
                    for line in buffer.replace('\r', '\n').split('\n'):
                        line = line.strip()
                        if not line:
                            continue
                        clean_line = re.sub(r'\x1b\[[0-9;]*[a-zA-Z]', '', line).strip()
                        if clean_line and ('%' not in clean_line):
                            sys.stdout.write(f"{clean_line}\n")
                            sys.stdout.flush()
                    break
                time.sleep(0.05)
                
            try:
                os.close(master)
            except:
                pass
                
            if process.poll() is None:
                process.terminate()
                process.wait()
                
            if process.returncode == 0:
                try:
                    os.sync()
                except:
                    pass
                    
                item["status"] = "Completed"
                successful += 1
                print(f"\n\033[92m[SUCCESS] Completed transfer {idx}/{len(gdrive_link_queue)} directly into '{dest_display}'!\033[0m")
            else:
                item["status"] = "Failed"
                failed.append((page_url, dest_display))
                print(f"\n\033[91m[FAILED] Link {idx} exited with error code {process.returncode}\033[0m")
            
            render_queue()
        
        try:
            os.sync()
        except:
            pass
            
        print(f"\n\033[95m{'='*60}\033[0m")
        print(f"\033[92m[SUMMARY] {successful}/{len(gdrive_link_queue)} item(s) transferred successfully!\033[0m")
        if failed:
            print("\033[91m[FAILED ITEMS]:\033[0m")
            for fl, d in failed:
                print(f"  - {fl} ({d})")
                
        print(f"\n\033[96m[GOOGLE DRIVE WEB SYNC TIP]:\033[0m")
        print("1. If you don't see the folder on https://drive.google.com immediately:")
        print("   - Press 'Shift + R' or 'F5' in your browser tab to refresh Google Drive.")
        print("2. Check your Google Account avatar in the top-right corner of Google Drive:")
        print("   - Make sure you are viewing the EXACT SAME Google Account that was authorized in Step 1!")
        print(f"\033[95m{'='*60}\033[0m")

    add_btn.disabled = False
    remove_btn.disabled = False
    clear_btn.disabled = False
    reset_btn.disabled = False
    start_btn.disabled = False
    link_input.disabled = False
    subfolder_input.disabled = False
    base_folder_input.disabled = False
    folder_dropdown.disabled = False
    refresh_folders_btn.disabled = False
    inner_dropdown.disabled = False
    inner_subfolder_input.disabled = False
    folders = get_existing_gdrive_folders()
    folder_dropdown.options = ["[ ➕ New Movie / Type Below ]"] + folders

add_btn.on_click(on_add_clicked)
link_input.on_submit(on_add_clicked)
remove_btn.on_click(on_remove_clicked)
clear_btn.on_click(on_clear_clicked)
reset_btn.on_click(on_reset_clicked)
start_btn.on_click(run_transfers)

buttons_row = widgets.HBox([add_btn, remove_btn, clear_btn, reset_btn], layout=widgets.Layout(margin='6px 0'))
start_row = widgets.HBox([start_btn], layout=widgets.Layout(margin='10px 0'))

step1_header = widgets.HTML("<div style='font-size: 13px; font-weight: 700; color: #1a73e8; margin: 4px 0 2px 0;'>🔗 Step 1: Google Drive Link (File or Folder) & Auto-Name</div>")
step2_header = widgets.HTML("<div style='font-size: 13px; font-weight: 700; color: #1a73e8; margin: 10px 0 2px 0;'>📁 Step 2: Destination Google Drive (Movie & Inside Subfolders)</div>")
step3_header = widgets.HTML("<div style='font-size: 12px; font-weight: 600; color: #70757a; margin: 8px 0 2px 0;'>⚙️ Base Google Drive Folder:</div>")

display(widgets.VBox([
    widgets.HTML("<h3 style='margin: 0 0 10px 0; color: #1a73e8;'>⚡ Google Drive to Google Drive Cloud Transfer Queue</h3>"),
    step1_header,
    link_row,
    step2_header,
    folder_select_row,
    subfolder_input,
    inner_dropdown,
    inner_subfolder_input,
    step3_header,
    base_folder_input,
    buttons_row,
    start_row,
    queue_output,
    status_output
]))

render_queue()


### Step 3.6: Instant Google Drive Explorer & Sync
Run this cell to immediately inspect all transferred files and folders in your Google Drive without opening the Drive web UI.

In [ ]:
#@title Check Transferred Files in Google Drive
import os

try:
    os.sync()
except:
    pass

base_gdrive = "GDrive_Transfers"
try:
    if 'base_folder_input' in globals() and base_folder_input.value.strip():
        base_gdrive = base_folder_input.value.strip()
except:
    pass

drive_candidates = ["/content/drive/MyDrive", "/content/drive/My Drive"]
mount_path = None
for p in drive_candidates:
    if os.path.exists(p):
        mount_path = p
        break

if not mount_path:
    print("\033[91m[ERROR] Google Drive is not mounted! Run Step 1 first.\033[0m")
else:
    target_path = os.path.join(mount_path, base_gdrive)
    print(f"\033[94m[EXPLORING] Destination: {target_path}\033[0m\n")
    if not os.path.exists(target_path):
        print(f"\033[93m[NOTE] The folder '{base_gdrive}' has not been created yet.\033[0m")
    else:
        found_any = False
        total_size = 0
        for root, dirs, files in os.walk(target_path):
            rel = os.path.relpath(root, target_path)
            depth = 0 if rel == "." else rel.count(os.sep) + 1
            indent = "  " * depth
            folder_name = os.path.basename(root) if rel != "." else base_gdrive
            print(f"{indent}\033[96m📁 {folder_name}/\033[0m")
            for f in sorted(files):
                found_any = True
                fp = os.path.join(root, f)
                try:
                    sz = os.path.getsize(fp) / (1024 * 1024)
                    total_size += sz
                    print(f"{indent}  \033[92m📄 {f}\033[0m ({sz:.2f} MB)")
                except:
                    print(f"{indent}  \033[92m📄 {f}\033[0m")
        if not found_any:
            print(f"\033[93m[EMPTY] Folder '{base_gdrive}' exists but contains no files yet.\033[0m")
        else:
            print(f"\n\033[92m[TOTAL SIZE] {total_size / 1024:.2f} GB across all files in '{base_gdrive}'\033[0m")


### Step 4: Force Sync & Disconnect (Optional)
Run this cell once all your transfers are complete to instantly flush all cached data to Google Drive servers (so they appear on your phone/browser immediately) and safely unmount the connection.

In [ ]:
from google.colab import drive
print("[INFO] Flushing cache and disconnecting Drive...")
drive.flush_and_unmount()
print("[SUCCESS] Google Drive successfully synced and unmounted!")
